# SEC Filing Text Analysis: 2022 Market Stress

**Owner:** Fatima Nasir  
**Project:** Market Stress Early-Warning System  
**Period:** 2022

## Goal

Collect, inspect, and preprocess SEC filings from selected S&P 500 companies to prepare financial text for sentiment analysis.

## Companies Selected

- NVIDIA (NVDA)
- Apple (AAPL)
- Alphabet (GOOGL)
- Microsoft (MSFT)
- Amazon (AMZN)

## Analysis Plan

1. Load and inspect the available market data.
2. Retrieve SEC company information and filing metadata.
3. Identify 2022 Form 10-K and Form 10-Q filings.
4. Download and inspect filing text.
5. Clean and organize the text for sentiment analysis.
6. Check data quality and identify limitations.

**Note:** This notebook focuses on SEC filing data preparation. Sentiment scoring will require an additional analysis step.

## 1. Setup and Imports

Import the Python libraries needed to retrieve SEC filing information, process company metadata, and prepare the filing text for analysis.

In [2]:

import os
import pandas as pd

print("Current folder:", os.getcwd())
print("\nChecking for the project's market data...")

file_path = r"C:\Users\LATITUDE\Downloads\RGBFS-main\RGBFS-main\sp500_historical_data.csv"

if os.path.exists(file_path):
    df = pd.read_csv(file_path)
    print("Dataset loaded successfully!")
    print("Shape:", df.shape)
    display(df.head())
else:
    print("CSV not found at:", file_path)

Current folder: c:\Users\LATITUDE\Downloads\RGBFS-main

Checking for the project's market data...
Dataset loaded successfully!
Shape: (5157, 6)


,Price,Close,High,Low,Open,Volume
0,Ticker,^GSPC,^GSPC,^GSPC,^GSPC,^GSPC
1,Date,NaN,NaN,NaN,NaN,NaN
2,1927-12-26,17.65999984741211,17.65999984741211,17.65999984741211,17.65999984741211,0
3,1928-01-02,17.65999984741211,17.760000228881836,17.549999237060547,17.760000228881836,0
4,1928-01-09,17.579999923706055,17.579999923706055,17.350000381469727,17.5,0


In [3]:

import pandas as pd

# Read the original CSV
raw_df = pd.read_csv(file_path, header=None)

# The first row contains Price, Close, High, Low, Open, Volume
# The second row contains Ticker information
# The third row contains Date and empty cells
# Actual market data starts on row 4

df = raw_df.iloc[3:].copy()

# Give the columns their correct names
df.columns = ["Date", "Close", "High", "Low", "Open", "Volume"]

# Clean the date column
df["Date"] = pd.to_datetime(df["Date"], errors="coerce")
df = df.dropna(subset=["Date"]).reset_index(drop=True)

# Convert prices and volume to numbers
for col in ["Close", "High", "Low", "Open", "Volume"]:
    df[col] = pd.to_numeric(df[col], errors="coerce")

print("Cleaned dataset shape:", df.shape)
print("\nColumn names:", df.columns.tolist())
display(df.head())


Cleaned dataset shape: (5155, 6)

Column names: ['Date', 'Close', 'High', 'Low', 'Open', 'Volume']


,Date,Close,High,Low,Open,Volume
0,1927-12-26,17.660000,17.660000,17.660000,17.660000,0
1,1928-01-02,17.660000,17.760000,17.549999,17.760000,0
2,1928-01-09,17.580000,17.580000,17.350000,17.500000,0
3,1928-01-16,17.480000,17.480000,17.260000,17.290001,0
4,1928-01-23,17.690001,17.709999,17.520000,17.639999,0


In [4]:

%pip install requests
%pip install beautifulsoup4

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.2.2
[notice] To update, run: python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.2.2
[notice] To update, run: python.exe -m pip install --upgrade pip


## 2. SEC Company Information and Filing Metadata

Retrieve SEC company information and identify the filings available for the selected companies.

The analysis focuses on five major technology companies: NVIDIA, Apple, Alphabet, Microsoft, and Amazon.

The filing metadata helps identify relevant Form 10-K (annual reports) and Form 10-Q (quarterly reports) filed during 2022. This information will be used to select documents for text preprocessing and future sentiment analysis.

In [5]:

import requests
import pandas as pd

# SEC requires a descriptive User-Agent
headers = {
    "User-Agent": "MarketStressStudentProject fnasir1@charlotte.edu"
}

# Download the SEC company ticker list
url = "https://www.sec.gov/files/company_tickers.json"

response = requests.get(url, headers=headers, timeout=30)

print("HTTP status:", response.status_code)

if response.status_code == 200:
    companies = pd.DataFrame.from_dict(
        response.json(), orient="index"
    )

    print("Company list downloaded!")
    display(companies.head())
else:
    print("SEC request failed. Check the status code above.")


HTTP status: 200
Company list downloaded!


,cik_str,ticker,title
0,1045810,NVDA,NVIDIA CORP
1,320193,AAPL,Apple Inc.
2,1652044,GOOGL,Alphabet Inc.
3,789019,MSFT,MICROSOFT CORP
4,1018724,AMZN,AMAZON COM INC


## 3. Selection of Sample Companies

Five major technology companies were selected for the SEC filing analysis: NVIDIA, Apple, Alphabet, Microsoft, and Amazon.

These companies provide a focused sample for examining financial disclosures during the 2022 market downturn.

The selected companies are filtered from the SEC company ticker dataset to obtain their names, ticker symbols, and Central Index Key (CIK) identifiers.

In [6]:
# Select five sample companies
sample_tickers = ["NVDA", "AAPL", "GOOGL", "MSFT", "AMZN"]

sample_companies = companies[
    companies["ticker"].isin(sample_tickers)
].copy()

display(sample_companies)


,cik_str,ticker,title
0,1045810,NVDA,NVIDIA CORP
1,320193,AAPL,Apple Inc.
2,1652044,GOOGL,Alphabet Inc.
3,789019,MSFT,MICROSOFT CORP
4,1018724,AMZN,AMAZON COM INC


## 4. Retrieving SEC Filing Records

SEC filing records are retrieved for the five selected companies using their Central Index Key (CIK) identifiers.

The SEC submissions data provides information about each filing, including the company ticker, filing date, form type, accession number, and primary document.

These records are collected into a structured dataset to support the identification and retrieval of relevant financial disclosures.

In [7]:

import time

filing_records = []

for _, company in sample_companies.iterrows():
    cik = int(company["cik_str"])
    ticker = company["ticker"]
    name = company["title"]

    url = f"https://data.sec.gov/submissions/CIK{cik:010d}.json"

    response = requests.get(url, headers=headers, timeout=30)

    print(ticker, response.status_code)

    if response.status_code == 200:
        data = response.json()
        recent = data["filings"]["recent"]

        for i, form in enumerate(recent["form"]):
            if form in ["10-K", "10-Q"]:
                filing_records.append({
                    "ticker": ticker,
                    "company": name,
                    "form": form,
                    "filing_date": recent["filingDate"][i],
                    "accession_number": recent["accessionNumber"][i],
                    "primary_document": recent["primaryDocument"][i]
                })

                if sum(r["ticker"] == ticker for r in filing_records) >= 4:
                    break

    time.sleep(0.15)

filings_df = pd.DataFrame(filing_records)

print("\nFiling records collected:", len(filings_df))
display(filings_df.head(20))


NVDA 200
AAPL 200
GOOGL 200
MSFT 200
AMZN 200

Filing records collected: 20


,ticker,company,form,filing_date,accession_number,primary_document
0,NVDA,NVIDIA CORP,10-Q,2026-08-26,0001045810-26-000075,nvda-20260726.htm
1,NVDA,NVIDIA CORP,10-Q,2026-05-20,0001045810-26-000052,nvda-20260426.htm
2,NVDA,NVIDIA CORP,10-K,2026-02-25,0001045810-26-000021,nvda-20260125.htm
3,NVDA,NVIDIA CORP,10-Q,2025-11-19,0001045810-25-000230,nvda-20251026.htm
4,AAPL,Apple Inc.,10-Q,2026-07-31,0000320193-26-000020,aapl-20260627.htm
5,AAPL,Apple Inc.,10-Q,2026-05-01,0000320193-26-000013,aapl-20260328.htm
6,AAPL,Apple Inc.,10-Q,2026-01-30,0000320193-26-000006,aapl-20251227.htm
7,AAPL,Apple Inc.,10-K,2025-10-31,0000320193-25-000079,aapl-20250927.htm
8,GOOGL,Alphabet Inc.,10-Q,2026-07-23,0001652044-26-000071,goog-20260630.htm
9,GOOGL,Alphabet Inc.,10-Q,2026-04-30,0001652044-26-000048,goog-20260331.htm


## 5. Inspecting Downloaded SEC Filings

This section examines the text files stored in the SEC filings directory.

The purpose is to check whether filing documents are available and inspect their contents before further processing.

This initial inspection helps identify potential issues with missing files, document formatting, and text quality that may affect subsequent analysis.

In [8]:
from pathlib import Path
text_folder = Path.cwd() / "data" / "sec_filings"

for file in sorted(text_folder.glob("*.txt")):
    text = file.read_text(encoding="utf-8")

    print("\nCompany file:", file.name)
    print("Characters:", len(text))
    print("First 500 characters:")
    print(text[:500])
    print("-" * 60)



Company file: AAPL_10-Q.txt
Characters: 95002
First 500 characters:
aapl-20260627 false 2026 Q3 0000320193 --09-26 P1Y P1Y P1Y P1Y http://fasb.org/us-gaap/2025#LongTermDebtCurrent http://fasb.org/us-gaap/2025#LongTermDebtNoncurrent http://fasb.org/us-gaap/2025#LongTermDebtCurrent http://fasb.org/us-gaap/2025#LongTermDebtNoncurrent P367D P156D xbrli:shares iso4217:USD iso4217:USD xbrli:shares xbrli:pure aapl:Customer aapl:Vendor 0000320193 2025-09-28 2026-06-27 0000320193 us-gaap:CommonStockMember 2025-09-28 2026-06-27 0000320193 aapl:A1.625NotesDue2026Member 202
------------------------------------------------------------

Company file: AMZN_10-Q.txt
Characters: 222671
First 500 characters:
amzn-20260630 false 2026 Q2 0001018724 12/31 P4Y0M P4Y0M http://fasb.org/us-gaap/2026#PropertyPlantAndEquipmentAndFinanceLeaseRightOfUseAssetAfterAccumulatedDepreciationAndAmortization http://fasb.org/us-gaap/2026#PropertyPlantAndEquipmentAndFinanceLeaseRightOfUseAssetAfterAccumulatedDepreciationAn

In [9]:

from pathlib import Path
import pandas as pd

# Create the folder if it does not already exist
output_folder = Path.cwd() / "data" / "sec_filings"
output_folder.mkdir(parents=True, exist_ok=True)

# Save the filing metadata
metadata_df = pd.DataFrame(filing_records)

metadata_path = output_folder / "filing_metadata.csv"
metadata_df.to_csv(metadata_path, index=False)

print("Metadata saved to:", metadata_path)
print("Number of records:", len(metadata_df))
print("Columns:", metadata_df.columns.tolist())

display(metadata_df.head())


Metadata saved to: c:\Users\LATITUDE\Downloads\RGBFS-main\data\sec_filings\filing_metadata.csv
Number of records: 20
Columns: ['ticker', 'company', 'form', 'filing_date', 'accession_number', 'primary_document']


,ticker,company,form,filing_date,accession_number,primary_document
0,NVDA,NVIDIA CORP,10-Q,2026-08-26,0001045810-26-000075,nvda-20260726.htm
1,NVDA,NVIDIA CORP,10-Q,2026-05-20,0001045810-26-000052,nvda-20260426.htm
2,NVDA,NVIDIA CORP,10-K,2026-02-25,0001045810-26-000021,nvda-20260125.htm
3,NVDA,NVIDIA CORP,10-Q,2025-11-19,0001045810-25-000230,nvda-20251026.htm
4,AAPL,Apple Inc.,10-Q,2026-07-31,0000320193-26-000020,aapl-20260627.htm


In [10]:

# Check the filing dates currently available
metadata_df["filing_date"] = pd.to_datetime(
    metadata_df["filing_date"], errors="coerce"
)

print("Earliest filing date:", metadata_df["filing_date"].min())
print("Latest filing date:", metadata_df["filing_date"].max())

# Count filings by year
print("\nFiling records by year:")
print(metadata_df["filing_date"].dt.year.value_counts().sort_index())

# Check whether any 2022 filings are already available
filings_2022 = metadata_df[
    metadata_df["filing_date"].dt.year == 2022
]

print("\nNumber of filings from 2022:", len(filings_2022))
display(filings_2022)


Earliest filing date: 2025-10-29 00:00:00
Latest filing date: 2026-08-26 00:00:00

Filing records by year:
filing_date
2025     5
2026    15
Name: count, dtype: int64

Number of filings from 2022: 0


,ticker,company,form,filing_date,accession_number,primary_document


## 6. Downloading SEC Filings for 2022

This section downloads the selected companies' SEC filing documents for the 2022 analysis period.

The downloaded filings provide the source text needed for financial disclosure analysis and future sentiment scoring.

In [35]:

import requests
import pandas as pd
from pathlib import Path
import time

headers = {
    "User-Agent": "MarketStressResearch Student fnasir1@charlotte.edu"
}

# SEC company identifiers for our sample
company_ciks = {
    "NVDA": "0001045810",
    "AAPL": "0000320193",
    "GOOGL": "0001652044",
    "MSFT": "0000789019",
    "AMZN": "0001018724"
}

historical_records = []

for ticker, cik in company_ciks.items():
    url = f"https://data.sec.gov/submissions/CIK{cik}.json"
    response = requests.get(url, headers=headers, timeout=30)

    print(ticker, "HTTP status:", response.status_code)

    if response.status_code != 200:
        continue

    data = response.json()
    recent = data["filings"]["recent"]

    # Include older SEC filing records when available
    all_filings = [recent]

    for old_file in data["filings"].get("files", []):
        old_url = "https://data.sec.gov/submissions/" + old_file["name"]
        old_response = requests.get(
            old_url, headers=headers, timeout=30
        )

        if old_response.status_code == 200:
            all_filings.append(old_response.json())

        time.sleep(0.2)

    company = data["name"]

    for filing_data in all_filings:
        forms = filing_data["form"]
        dates = filing_data["filingDate"]
        accessions = filing_data["accessionNumber"]
        documents = filing_data["primaryDocument"]

        for form, date, accession, document in zip(
            forms, dates, accessions, documents
        ):
            if (
                form in ["10-K", "10-Q"]
                and "2022-01-01" <= date <= "2022-12-31"
            ):
                historical_records.append({
                    "ticker": ticker,
                    "company": company,
                    "form": form,
                    "filing_date": date,
                    "accession_number": accession,
                    "primary_document": document
                })

    time.sleep(0.2)

historical_df = pd.DataFrame(historical_records)

print("\nHistorical filings found:", len(historical_df))
display(historical_df.head(25))

output_folder = Path.cwd() / "data" / "sec_filings"
output_folder.mkdir(parents=True, exist_ok=True)

historical_path = output_folder / "filing_metadata_2022.csv"
historical_df.to_csv(historical_path, index=False)

print("\nSaved to:", historical_path)


NVDA HTTP status: 200
AAPL HTTP status: 200
GOOGL HTTP status: 200
MSFT HTTP status: 200
AMZN HTTP status: 200

Historical filings found: 20


,ticker,company,form,filing_date,accession_number,primary_document
0,NVDA,NVIDIA CORP,10-Q,2022-11-18,0001045810-22-000166,nvda-20221030.htm
1,NVDA,NVIDIA CORP,10-Q,2022-08-31,0001045810-22-000147,nvda-20220731.htm
2,NVDA,NVIDIA CORP,10-Q,2022-05-27,0001045810-22-000079,nvda-20220501.htm
3,NVDA,NVIDIA CORP,10-K,2022-03-18,0001045810-22-000036,nvda-20220130.htm
4,AAPL,Apple Inc.,10-K,2022-10-28,0000320193-22-000108,aapl-20220924.htm
5,AAPL,Apple Inc.,10-Q,2022-07-29,0000320193-22-000070,aapl-20220625.htm
6,AAPL,Apple Inc.,10-Q,2022-04-29,0000320193-22-000059,aapl-20220326.htm
7,AAPL,Apple Inc.,10-Q,2022-01-28,0000320193-22-000007,aapl-20211225.htm
8,GOOGL,Alphabet Inc.,10-Q,2022-10-26,0001652044-22-000090,goog-20220930.htm
9,GOOGL,Alphabet Inc.,10-Q,2022-07-27,0001652044-22-000071,goog-20220630.htm



Saved to: c:\Users\LATITUDE\Downloads\RGBFS-main\data\sec_filings\filing_metadata_2022.csv


In [12]:

import requests
from bs4 import BeautifulSoup
from pathlib import Path

# Select the first 2022 filing
row = historical_df.iloc[0]

# Build the SEC filing URL
accession_clean = row["accession_number"].replace("-", "")
cik = str(int(row["accession_number"].split("-")[0]))
document = row["primary_document"]

url = (
    f"https://www.sec.gov/Archives/edgar/data/"
    f"{cik}/{accession_clean}/{document}"
)

headers = {
    "User-Agent": "MarketStressResearch Student fnasir1@charlotte.edu"
}

response = requests.get(url, headers=headers, timeout=30)

print("Company:", row["company"])
print("Filing date:", row["filing_date"])
print("HTTP status:", response.status_code)

if response.status_code == 200:
    soup = BeautifulSoup(response.content, "html.parser")

    for tag in soup(["script", "style"]):
        tag.decompose()

    text = soup.get_text(" ", strip=True)

    print("Extracted characters:", len(text))
    print("\nText preview:\n", text[:1500])

    # Save this test document separately
    output_folder = Path.cwd() / "data" / "sec_filings_2022"
    output_folder.mkdir(parents=True, exist_ok=True)

    output_file = output_folder / f"{row['ticker']}_{row['filing_date']}.txt"
    output_file.write_text(text, encoding="utf-8")

    print("\nSaved test filing to:", output_file)
else:
    print("Document retrieval failed. Existing files were not changed.")
    print(response.text[:300])


Company: NVIDIA CORP
Filing date: 2022-11-18
HTTP status: 200
Extracted characters: 149449

Text preview:
 nvda-20221030 0001045810 1/29 2023 Q3 FALSE 0001045810 2022-01-31 2022-10-30 0001045810 2022-11-11 xbrli:shares 0001045810 2022-08-01 2022-10-30 iso4217:USD 0001045810 2021-08-02 2021-10-31 0001045810 2021-02-01 2021-10-31 iso4217:USD xbrli:shares 0001045810 2022-10-30 0001045810 2022-01-30 0001045810 us-gaap:CommonStockMember 2022-07-31 0001045810 us-gaap:AdditionalPaidInCapitalMember 2022-07-31 0001045810 us-gaap:TreasuryStockMember 2022-07-31 0001045810 us-gaap:AccumulatedOtherComprehensiveIncomeMember 2022-07-31 0001045810 us-gaap:RetainedEarningsMember 2022-07-31 0001045810 2022-07-31 0001045810 us-gaap:RetainedEarningsMember 2022-08-01 2022-10-30 0001045810 us-gaap:AccumulatedOtherComprehensiveIncomeMember 2022-08-01 2022-10-30 0001045810 us-gaap:CommonStockMember 2022-08-01 2022-10-30 0001045810 us-gaap:AdditionalPaidInCapitalMember 2022-08-01 2022-10-30 0001045810 us-gaap:

In [36]:

import requests
import pandas as pd
from pathlib import Path
from bs4 import BeautifulSoup
import time

# Create folder for 2022 SEC filing text files
output_folder = Path.cwd() / "data" / "sec_filings_2022"
output_folder.mkdir(parents=True, exist_ok=True)

headers = {
    "User-Agent": "MarketStressResearch Student fnasir1@charlotte.edu"
}

downloaded = []

for _, row in historical_df.iterrows():

    filename = (
        f"{row['ticker']}_{row['filing_date']}_{row['form']}.txt"
    )

    file_path = output_folder / filename

    # Skip files that have already been downloaded
    if file_path.exists() and file_path.stat().st_size > 0:
        print("Already exists, skipping:", filename)
        continue

    accession = row["accession_number"].replace("-", "")
    cik = str(int(row["accession_number"].split("-")[0]))
    document = row["primary_document"]

    url = (
        f"https://www.sec.gov/Archives/edgar/data/"
        f"{cik}/{accession}/{document}"
    )

    response = requests.get(url, headers=headers, timeout=30)

    if response.status_code == 200:
        soup = BeautifulSoup(response.content, "html.parser")

        for tag in soup(["script", "style"]):
            tag.decompose()

        text = soup.get_text(" ", strip=True)

        (output_folder / filename).write_text(
            text, encoding="utf-8"
        )

        downloaded.append({
            "ticker": row["ticker"],
            "filing_date": row["filing_date"],
            "form": row["form"],
            "characters": len(text),
            "file": filename
        })

        print("Downloaded:", filename, "| Characters:", len(text))

    else:
        print(
            "Failed:", row["ticker"], row["filing_date"],
            "| HTTP", response.status_code
        )

    time.sleep(0.2)

download_report = pd.DataFrame(downloaded)

print(
    "\nNewly downloaded:",
    len(download_report),
    "of",
    len(historical_df),
    "filings"
)

display(download_report)


Already exists, skipping: NVDA_2022-11-18_10-Q.txt
Already exists, skipping: NVDA_2022-08-31_10-Q.txt
Already exists, skipping: NVDA_2022-05-27_10-Q.txt
Already exists, skipping: NVDA_2022-03-18_10-K.txt
Already exists, skipping: AAPL_2022-10-28_10-K.txt
Already exists, skipping: AAPL_2022-07-29_10-Q.txt
Already exists, skipping: AAPL_2022-04-29_10-Q.txt
Already exists, skipping: AAPL_2022-01-28_10-Q.txt
Downloaded: GOOGL_2022-10-26_10-Q.txt | Characters: 184254
Downloaded: GOOGL_2022-07-27_10-Q.txt | Characters: 173571
Downloaded: GOOGL_2022-04-27_10-Q.txt | Characters: 149167
Downloaded: GOOGL_2022-02-02_10-K.txt | Characters: 363387
Already exists, skipping: MSFT_2022-10-25_10-Q.txt
Already exists, skipping: MSFT_2022-07-28_10-K.txt
Already exists, skipping: MSFT_2022-04-26_10-Q.txt
Already exists, skipping: MSFT_2022-01-25_10-Q.txt
Already exists, skipping: AMZN_2022-10-28_10-Q.txt
Already exists, skipping: AMZN_2022-07-29_10-Q.txt
Already exists, skipping: AMZN_2022-04-29_10-Q.txt

,ticker,filing_date,form,characters,file
0,GOOGL,2022-10-26,10-Q,184254,GOOGL_2022-10-26_10-Q.txt
1,GOOGL,2022-07-27,10-Q,173571,GOOGL_2022-07-27_10-Q.txt
2,GOOGL,2022-04-27,10-Q,149167,GOOGL_2022-04-27_10-Q.txt
3,GOOGL,2022-02-02,10-K,363387,GOOGL_2022-02-02_10-K.txt


In [14]:

msft_rows = historical_df[historical_df["ticker"] == "MSFT"]

display(
    msft_rows[
        ["ticker", "filing_date", "form",
         "accession_number", "primary_document"]
    ]
)


,ticker,filing_date,form,accession_number,primary_document
8,MSFT,2022-10-25,10-Q,0001564590-22-035087,msft-10q_20220930.htm
9,MSFT,2022-07-28,10-K,0001564590-22-026876,msft-10k_20220630.htm
10,MSFT,2022-04-26,10-Q,0001564590-22-015675,msft-10q_20220331.htm
11,MSFT,2022-01-25,10-Q,0001564590-22-002324,msft-10q_20211231.htm


In [15]:

import requests

row = msft_rows.iloc[0]

accession = row["accession_number"].replace("-", "")
document = row["primary_document"]

# Microsoft's CIK, without leading zeros in the URL
cik = "789019"

url = (
    f"https://www.sec.gov/Archives/edgar/data/"
    f"{cik}/{accession}/{document}"
)

print("URL:", url)

response = requests.get(
    url,
    headers=headers,
    timeout=30
)

print("HTTP status:", response.status_code)
print("Content type:", response.headers.get("Content-Type"))
print(response.text[:300] if response.status_code != 200 else "Document retrieved!")


URL: https://www.sec.gov/Archives/edgar/data/789019/000156459022035087/msft-10q_20220930.htm
HTTP status: 200
Content type: text/html
Document retrieved!


In [16]:

from pathlib import Path
import requests
from bs4 import BeautifulSoup
import time

output_folder = Path.cwd() / "data" / "sec_filings_2022"
output_folder.mkdir(parents=True, exist_ok=True)

headers = {
    "User-Agent": "MarketStressResearch Student fnasir1@charlotte.edu"
}

msft_rows = historical_df[historical_df["ticker"] == "MSFT"]

for _, row in msft_rows.iterrows():
    accession = row["accession_number"].replace("-", "")
    document = row["primary_document"]
    cik = "789019"

    url = (
        f"https://www.sec.gov/Archives/edgar/data/"
        f"{cik}/{accession}/{document}"
    )

    response = requests.get(url, headers=headers, timeout=30)

    if response.status_code == 200:
        soup = BeautifulSoup(response.content, "html.parser")

        for tag in soup(["script", "style"]):
            tag.decompose()

        text = soup.get_text(" ", strip=True)

        filename = (
            f"{row['ticker']}_{row['filing_date']}_{row['form']}.txt"
        )
        (output_folder / filename).write_text(text, encoding="utf-8")

        print("Downloaded:", filename, "| Characters:", len(text))
    else:
        print("Failed:", row["filing_date"], "HTTP", response.status_code)

    time.sleep(0.2)


Downloaded: MSFT_2022-10-25_10-Q.txt | Characters: 216167
Downloaded: MSFT_2022-07-28_10-K.txt | Characters: 378985
Downloaded: MSFT_2022-04-26_10-Q.txt | Characters: 245088
Downloaded: MSFT_2022-01-25_10-Q.txt | Characters: 237172


In [17]:

from pathlib import Path

folder = Path.cwd() / "data" / "sec_filings_2022"

files = sorted(folder.glob("*.txt"))

print("Total filing files:", len(files))

for file in files:
    text = file.read_text(encoding="utf-8", errors="ignore")

    print("\nFile:", file.name)
    print("Characters:", len(text))
    print("Contains XBRL markers:", "xbrli:" in text or "us-gaap:" in text)
    print("Preview:", text[:300])


Total filing files: 17

File: AAPL_2022-01-28_10-Q.txt
Characters: 67389
Contains XBRL markers: True
Preview: aapl-20211225 false 2022 Q1 0000320193 --09-24 P1Y 64 P1Y 27 P1Y 8 1 0000320193 2021-09-26 2021-12-25 0000320193 us-gaap:CommonStockMember 2021-09-26 2021-12-25 0000320193 aapl:A1.000NotesDue2022Member 2021-09-26 2021-12-25 0000320193 aapl:A1.375NotesDue2024Member 2021-09-26 2021-12-25 0000320193 aa

File: AAPL_2022-04-29_10-Q.txt
Characters: 75140
Contains XBRL markers: True
Preview: aapl-20220326 false 2022 Q2 0000320193 --09-24 P1Y 63 P1Y 27 P1Y 8 2 0000320193 2021-09-26 2022-03-26 0000320193 us-gaap:CommonStockMember 2021-09-26 2022-03-26 0000320193 aapl:A1.000NotesDue2022Member 2021-09-26 2022-03-26 0000320193 aapl:A1.375NotesDue2024Member 2021-09-26 2022-03-26 0000320193 aa

File: AAPL_2022-07-29_10-Q.txt
Characters: 73244
Contains XBRL markers: True
Preview: aapl-20220625 false 2022 Q3 0000320193 --09-24 P1Y 63 P1Y 27 P1Y 8 2 0000320193 2021-09-26 2022-06-25 0000320193 u

In [18]:

from pathlib import Path
import re

folder = Path.cwd() / "data" / "sec_filings_2022"

# Use one Microsoft filing as a test
file = folder / "MSFT_2022-10-25_10-Q.txt"
text = file.read_text(encoding="utf-8", errors="ignore")

# Find likely narrative sections
for term in [
    "Management's Discussion and Analysis",
    "MANAGEMENT'S DISCUSSION AND ANALYSIS",
    "Risk Factors",
    "RISK FACTORS"
]:
    match = re.search(term, text, flags=re.IGNORECASE)
    print(term, "->", match.start() if match else "not found")

print("\nText sample around first Risk Factors heading:")
match = re.search("Risk Factors", text, flags=re.IGNORECASE)

if match:
    print(text[match.start():match.start() + 1500])
else:
    print(text[:1500])


Management's Discussion and Analysis -> not found
MANAGEMENT'S DISCUSSION AND ANALYSIS -> not found
Risk Factors -> 27693
RISK FACTORS -> 27693

Text sample around first Risk Factors heading:
Risk Factors 46 Item 2. Unregistered Sales of Equity Securities and Use of Proceeds 61 Item 6. Exhibits 62 SIGNATURE 63 2 PART I Item 1 PART I. FINANCIAL INFORMATION ITEM 1. FINANCIAL STATEMENTS INCOME STATEMENTS (In millions, except per share amounts) (Unaudited) Three Months Ended September 30, 2022 2021 Revenue: Product $ 15,741 $ 16,631 Service and other 34,381 28,686 Total revenue 50,122 45,317 Cost of revenue: Product 4,302 3,792 Service and other 11,150 9,854 Total cost of revenue 15,452 13,646 Gross margin 34,670 31,671 Research and development 6,628 5,599 Sales and marketing 5,126 4,547 General and administrative 1,398 1,287 Operating income 21,518 20,238 Other income, net 54 286 Income before income taxes 21,572 20,524 Provision for income taxes 4,016 19 Net income $ 17,556 $ 20,505 Earn

In [19]:

import requests
from bs4 import BeautifulSoup
from pathlib import Path

row = historical_df[
    (historical_df["ticker"] == "MSFT") &
    (historical_df["filing_date"] == "2022-10-25")
].iloc[0]

cik = "789019"
accession = row["accession_number"].replace("-", "")
document = row["primary_document"]

url = (
    f"https://www.sec.gov/Archives/edgar/data/"
    f"{cik}/{accession}/{document}"
)

response = requests.get(url, headers=headers, timeout=30)

print("HTTP status:", response.status_code)

if response.status_code == 200:
    soup = BeautifulSoup(response.content, "html.parser")

    # Inspect the original HTML for the filing's narrative headings
    page_text = soup.get_text(" ", strip=True)

    for heading in [
        "Management's Discussion and Analysis",
        "Risk Factors",
        "Item 2"
    ]:
        print(heading, "found:", heading.lower() in page_text.lower())

    print("\nOriginal HTML text preview:\n", page_text[:1200])
else:
    print(response.text[:300])


HTTP status: 200
Management's Discussion and Analysis found: False
Risk Factors found: True
Item 2 found: True

Original HTML text preview:
 msft-10q_20220930.htm false 2023 Q1 0000789019 --06-30 http://fasb.org/us-gaap/2022#OtherAssetsCurrent http://fasb.org/us-gaap/2022#OtherAssetsCurrent http://fasb.org/us-gaap/2022#OtherAssetsCurrent http://fasb.org/us-gaap/2022#OtherAssetsCurrent http://fasb.org/us-gaap/2022#OtherAssetsNoncurrent http://fasb.org/us-gaap/2022#OtherAssetsNoncurrent http://fasb.org/us-gaap/2022#OtherLiabilitiesCurrent http://fasb.org/us-gaap/2022#OtherLiabilitiesCurrent http://fasb.org/us-gaap/2022#OtherLiabilitiesNoncurrent http://fasb.org/us-gaap/2022#OtherLiabilitiesNoncurrent 0000789019 2022-07-01 2022-09-30 0000789019 us-gaap:CommonStockMember 2022-07-01 2022-09-30 xbrli:shares 0000789019 2022-10-20 iso4217:USD xbrli:shares 0000789019 msft:NotesThreePointOneTwoFivePercentDueDecemberSixTwentyTwentyEightMember 2022-07-01 2022-09-30 0000789019 msft:NotesTwoPointSix

In [20]:

# Inspect HTML elements containing the Risk Factors heading
from bs4 import BeautifulSoup
import re

if response.status_code == 200:
    soup = BeautifulSoup(response.content, "html.parser")

    matches = []

    for element in soup.find_all(
        string=re.compile(r"Risk Factors", re.IGNORECASE)
    ):
        parent = element.parent

        matches.append({
            "tag": parent.name,
            "text": parent.get_text(" ", strip=True)[:250]
        })

    print("Risk Factors matches:", len(matches))

    for item in matches[:10]:
        print(item)
else:
    print("The SEC document was not retrieved successfully.")


Risk Factors matches: 6
{'tag': 'span', 'text': 'Risk Factors'}
{'tag': 'p', 'text': 'This report includes estimates, projections, statements relating to our business plans, objectives, and expected operating results that are “forward-looking statements” within the meaning of the Private Securities Litigation Reform Act of 1995, Secti'}
{'tag': 'p', 'text': 'Our international operations provide a significant portion of our total revenue and expenses. Many of these revenue and expenses are denominated in currencies other than the U.S. dollar. As a result, changes in foreign exchange rates may significantl'}
{'tag': 'p', 'text': 'ITEM\xa01A. RISK FACTORS'}
{'tag': 'span', 'text': 'Source code, the detailed program commands for our operating systems and other software programs, is critical to our business. Although we license portions of our application and operating system source code to several licensees, we take significant '}
{'tag': 'p', 'text': 'Measures to contain a global pandemic

In [21]:

from bs4 import BeautifulSoup
import re

if response.status_code == 200:
    soup = BeautifulSoup(response.content, "html.parser")

    # Find paragraphs containing substantive risk discussion
    risk_paragraphs = []

    for paragraph in soup.find_all("p"):
        text = paragraph.get_text(" ", strip=True)

        if (
            len(text.split()) >= 20
            and re.search(
                r"risk|uncertain|adverse|exposure|may affect",
                text,
                flags=re.IGNORECASE
            )
        ):
            risk_paragraphs.append(text)

    print("Risk-related paragraphs found:", len(risk_paragraphs))

    for paragraph in risk_paragraphs[:5]:
        print("\n-", paragraph[:500])
else:
    print("Please retrieve the original HTML successfully first.")


Risk-related paragraphs found: 63

- Preparing financial statements requires management to make estimates and assumptions that affect the reported amounts of assets, liabilities, revenue, and expenses. Examples of estimates and assumptions include: for revenue recognition, determining the nature and timing of satisfaction of performance obligations, and determining the standalone selling price (“SSP”) of performance obligations, variable consideration, and other obligations such as product returns and refunds; loss contingencies; p

- Debt investments are classified as available-for-sale and realized gains and losses are recorded using the specific identification method. Changes in fair value, excluding credit losses and impairments, are recorded in other comprehensive income . Fair value is calculated based on publicly available market information or other estimates determined by management. If the cost of an investment exceeds its fair value, we evaluate, among other factors, general

In [22]:

from pathlib import Path
import pandas as pd

output_folder = Path.cwd() / "data" / "sec_filings_2022"
output_folder.mkdir(parents=True, exist_ok=True)

# Save the paragraphs found in the Microsoft test filing
paragraphs_df = pd.DataFrame({
    "paragraph_id": range(1, len(risk_paragraphs) + 1),
    "text": risk_paragraphs
})

output_file = output_folder / "MSFT_2022-10-25_risk_paragraphs.csv"
paragraphs_df.to_csv(output_file, index=False)

print("Paragraphs saved:", len(paragraphs_df))
print("Saved to:", output_file)

display(paragraphs_df.head(5))


Paragraphs saved: 63
Saved to: c:\Users\LATITUDE\Downloads\RGBFS-main\data\sec_filings_2022\MSFT_2022-10-25_risk_paragraphs.csv


,paragraph_id,text
0,1,Preparing financial statements requires manage...
1,2,Debt investments are classified as available-f...
2,3,We use derivative instruments to manage risks ...
3,4,"Certain forecasted transactions, assets, and l..."
4,5,Foreign currency risks related to certain non-...


In [23]:

import requests
import pandas as pd
from bs4 import BeautifulSoup
from pathlib import Path
import time

output_folder = Path.cwd() / "data" / "sec_filings_2022"
output_folder.mkdir(parents=True, exist_ok=True)

headers = {
    "User-Agent": "MarketStressResearch Student fnasir1@charlotte.edu"
}

all_paragraphs = []
failed_filings = []

for _, row in historical_df.iterrows():
    accession = row["accession_number"].replace("-", "")
    cik = str(int(row["accession_number"].split("-")[0]))
    document = row["primary_document"]

    url = (
        f"https://www.sec.gov/Archives/edgar/data/"
        f"{cik}/{accession}/{document}"
    )

    response = requests.get(url, headers=headers, timeout=30)

    if response.status_code == 200:
        soup = BeautifulSoup(response.content, "html.parser")

        for tag in soup(["script", "style"]):
            tag.decompose()

        for paragraph in soup.find_all("p"):
            text = paragraph.get_text(" ", strip=True)

            # Keep paragraphs long enough to be potentially useful
            if len(text.split()) >= 20:
                all_paragraphs.append({
                    "ticker": row["ticker"],
                    "company": row["company"],
                    "filing_date": row["filing_date"],
                    "form": row["form"],
                    "text": text
                })

        print("Processed:", row["ticker"], row["filing_date"])
    else:
        failed_filings.append({
            "ticker": row["ticker"],
            "filing_date": row["filing_date"],
            "status": response.status_code
        })
        print("Failed:", row["ticker"], row["filing_date"],
              response.status_code)

    time.sleep(0.2)

paragraphs_df = pd.DataFrame(all_paragraphs)

output_file = output_folder / "sec_paragraphs_2022.csv"
paragraphs_df.to_csv(output_file, index=False)

print("\nTotal paragraphs:", len(paragraphs_df))
print("Failed filings:", len(failed_filings))
print("Saved to:", output_file)

display(paragraphs_df.head())
if failed_filings:
    display(pd.DataFrame(failed_filings))


Processed: NVDA 2022-11-18
Processed: NVDA 2022-08-31
Processed: NVDA 2022-05-27
Processed: NVDA 2022-03-18
Processed: AAPL 2022-10-28
Processed: AAPL 2022-07-29
Processed: AAPL 2022-04-29
Processed: AAPL 2022-01-28
Failed: MSFT 2022-10-25 404
Failed: MSFT 2022-07-28 404
Failed: MSFT 2022-04-26 404
Failed: MSFT 2022-01-25 404
Processed: AMZN 2022-10-28
Processed: AMZN 2022-07-29
Processed: AMZN 2022-04-29
Processed: AMZN 2022-02-04

Total paragraphs: 0
Failed filings: 4
Saved to: c:\Users\LATITUDE\Downloads\RGBFS-main\data\sec_filings_2022\sec_paragraphs_2022.csv


""


,ticker,filing_date,status
0,MSFT,2022-10-25,404
1,MSFT,2022-07-28,404
2,MSFT,2022-04-26,404
3,MSFT,2022-01-25,404


In [24]:

from pathlib import Path

folder = Path.cwd() / "data" / "sec_filings_2022"

files = sorted(folder.glob("*.txt"))

print("Existing downloaded text files:", len(files))

for file in files:
    print(file.name, "-", file.stat().st_size, "bytes")


Existing downloaded text files: 17
AAPL_2022-01-28_10-Q.txt - 68089 bytes
AAPL_2022-04-29_10-Q.txt - 75857 bytes
AAPL_2022-07-29_10-Q.txt - 73920 bytes
AAPL_2022-10-28_10-K.txt - 235245 bytes
AMZN_2022-02-04_10-K.txt - 280044 bytes
AMZN_2022-04-29_10-Q.txt - 174920 bytes
AMZN_2022-07-29_10-Q.txt - 188287 bytes
AMZN_2022-10-28_10-Q.txt - 191122 bytes
MSFT_2022-01-25_10-Q.txt - 238187 bytes
MSFT_2022-04-26_10-Q.txt - 246067 bytes
MSFT_2022-07-28_10-K.txt - 380429 bytes
MSFT_2022-10-25_10-Q.txt - 216999 bytes
NVDA_2022-03-18_10-K.txt - 326711 bytes
NVDA_2022-05-27_10-Q.txt - 127146 bytes
NVDA_2022-08-31_10-Q.txt - 168634 bytes
NVDA_2022-11-18.txt - 150292 bytes
NVDA_2022-11-18_10-Q.txt - 150292 bytes


## 7. SEC Filing Data Quality Checks

This section checks whether the expected SEC filing text files are available for the selected companies.

The validation process identifies missing documents and helps assess the completeness of the collected dataset before further analysis.

In [25]:

from pathlib import Path

folder = Path.cwd() / "data" / "sec_filings_2022"

expected_files = []

for _, row in historical_df.iterrows():
    filename = (
        f"{row['ticker']}_{row['filing_date']}_{row['form']}.txt"
    )
    path = folder / filename

    expected_files.append({
        "ticker": row["ticker"],
        "filing_date": row["filing_date"],
        "form": row["form"],
        "file": filename,
        "exists": path.exists(),
        "bytes": path.stat().st_size if path.exists() else 0
    })

file_check = pd.DataFrame(expected_files)

print("Expected filings:", len(file_check))
print("Files found:", file_check["exists"].sum())
print("Files missing:", (~file_check["exists"]).sum())

display(file_check)


Expected filings: 16
Files found: 16
Files missing: 0


,ticker,filing_date,form,file,exists,bytes
0,NVDA,2022-11-18,10-Q,NVDA_2022-11-18_10-Q.txt,True,150292
1,NVDA,2022-08-31,10-Q,NVDA_2022-08-31_10-Q.txt,True,168634
2,NVDA,2022-05-27,10-Q,NVDA_2022-05-27_10-Q.txt,True,127146
3,NVDA,2022-03-18,10-K,NVDA_2022-03-18_10-K.txt,True,326711
4,AAPL,2022-10-28,10-K,AAPL_2022-10-28_10-K.txt,True,235245
5,AAPL,2022-07-29,10-Q,AAPL_2022-07-29_10-Q.txt,True,73920
6,AAPL,2022-04-29,10-Q,AAPL_2022-04-29_10-Q.txt,True,75857
7,AAPL,2022-01-28,10-Q,AAPL_2022-01-28_10-Q.txt,True,68089
8,MSFT,2022-10-25,10-Q,MSFT_2022-10-25_10-Q.txt,True,216999
9,MSFT,2022-07-28,10-K,MSFT_2022-07-28_10-K.txt,True,380429


## 8. SEC Filing Text Cleaning and Preprocessing

This section prepares the downloaded SEC filing documents for text analysis.

SEC filings may contain HTML tags, formatting elements, and other content that is not useful for natural language processing.

The preprocessing stage focuses on extracting readable text and improving its quality for future financial sentiment analysis.

In [37]:

from pathlib import Path
import pandas as pd
import re

folder = Path.cwd() / "data" / "sec_filings_2022"
cleaned_paragraphs = []

for _, row in historical_df.iterrows():
    filename = (
        f"{row['ticker']}_{row['filing_date']}_{row['form']}.txt"
    )
    file_path = folder / filename
    text = file_path.read_text(encoding="utf-8", errors="ignore")

    # Remove URLs and common XBRL tags
    text = re.sub(r"https?://\S+", " ", text)
    text = re.sub(r"\b(?:us-gaap|dei|xbrli|iso4217):[A-Za-z0-9_.-]+", " ", text)

    # Normalize whitespace
    text = re.sub(r"\s+", " ", text).strip()

    # Split into sentence-like segments for inspection
    segments = re.split(r"(?<=[.!?])\s+", text)

    for segment in segments:
        segment = segment.strip()

        if len(segment.split()) >= 15:
            cleaned_paragraphs.append({
                "ticker": row["ticker"],
                "filing_date": row["filing_date"],
                "form": row["form"],
                "text": segment
            })

cleaned_df = pd.DataFrame(cleaned_paragraphs)

output_file = folder / "sec_text_segments_2022.csv"
cleaned_df.to_csv(output_file, index=False)

print("Segments created:", len(cleaned_df))
print("Unique companies:", cleaned_df["ticker"].nunique())
print("Saved to:", output_file)

display(cleaned_df.head(10))


Segments created: 13217
Unique companies: 5
Saved to: c:\Users\LATITUDE\Downloads\RGBFS-main\data\sec_filings_2022\sec_text_segments_2022.csv


,ticker,filing_date,form,text
0,NVDA,2022-11-18,10-Q,nvda-20221030 0001045810 1/29 2023 Q3 FALSE 00...
1,NVDA,2022-11-18,10-Q,20549 FORM 10-Q ☒ QUARTERLY REPORT PURSUANT TO...
2,NVDA,2022-11-18,10-Q,Employer Incorporation or Organization) Identi...
3,NVDA,2022-11-18,10-Q,Yes ☒ No ☐ Indicate by check mark whether the ...
4,NVDA,2022-11-18,10-Q,Yes ☒ No ☐ Indicate by check mark whether the ...
5,NVDA,2022-11-18,10-Q,"See definitions of “large accelerated filer”, ..."
6,NVDA,2022-11-18,10-Q,Large accelerated filer ☒ Accelerated filer ☐ ...
7,NVDA,2022-11-18,10-Q,☐ Indicate by check mark whether the registran...
8,NVDA,2022-11-18,10-Q,Yes ☐ No ☒ The number of shares of common stoc...
9,NVDA,2022-11-18,10-Q,NVIDIA CORPORATION FORM 10-Q FOR THE QUARTER E...


In [38]:

print("Total text segments:", len(cleaned_df))
print("Companies:", cleaned_df["ticker"].nunique())
print("Missing text values:", cleaned_df["text"].isna().sum())
print("Empty text segments:", cleaned_df["text"].fillna("").str.strip().eq("").sum())

print("\nSegments by company:")
print(cleaned_df["ticker"].value_counts())


Total text segments: 13217
Companies: 5
Missing text values: 0
Empty text segments: 0

Segments by company:
ticker
MSFT     3881
NVDA     2698
GOOGL    2662
AMZN     2598
AAPL     1378
Name: count, dtype: int64


In [31]:

print("Companies in dataset:")
print(sorted(cleaned_df["ticker"].unique()))

print("\nNumber of segments by company:")
print(cleaned_df.groupby("ticker").size())

print("\nNumber of segments by filing:")
print(
    cleaned_df.groupby(["ticker", "filing_date", "form"])
    .size()
    .reset_index(name="segments")
    .to_string(index=False)
)


Companies in dataset:
['AAPL', 'AMZN', 'MSFT', 'NVDA']

Number of segments by company:
ticker
AAPL    1378
AMZN    2598
MSFT    3881
NVDA    2698
dtype: int64

Number of segments by filing:
ticker filing_date form  segments
  AAPL  2022-01-28 10-Q       173
  AAPL  2022-04-29 10-Q       185
  AAPL  2022-07-29 10-Q       177
  AAPL  2022-10-28 10-K       843
  AMZN  2022-02-04 10-K       928
  AMZN  2022-04-29 10-Q       547
  AMZN  2022-07-29 10-Q       558
  AMZN  2022-10-28 10-Q       565
  MSFT  2022-01-25 10-Q       835
  MSFT  2022-04-26 10-Q       856
  MSFT  2022-07-28 10-K      1413
  MSFT  2022-10-25 10-Q       777
  NVDA  2022-03-18 10-K      1207
  NVDA  2022-05-27 10-Q       439
  NVDA  2022-08-31 10-Q       552
  NVDA  2022-11-18 10-Q       500


In [34]:
print(metadata_df.loc[
    metadata_df["ticker"] == "GOOGL",
    ["ticker", "filing_date", "form"]
].to_string(index=False))

ticker filing_date form
 GOOGL  2026-07-23 10-Q
 GOOGL  2026-04-30 10-Q
 GOOGL  2026-02-05 10-K
 GOOGL  2025-10-30 10-Q


In [28]:

from pathlib import Path

folder = Path.cwd() / "data" / "sec_filings_2022"

print("Google filing files:")

for file in sorted(folder.glob("GOOGL*.txt")):
    print(file.name, "-", file.stat().st_size, "bytes")

print("\nGoogle rows in historical metadata:")
display(historical_df[historical_df["ticker"] == "GOOGL"])


Google filing files:

Google rows in historical metadata:


,ticker,company,form,filing_date,accession_number,primary_document


In [29]:

import requests
import pandas as pd
import time
from pathlib import Path

headers = {
    "User-Agent": "MarketStressResearch Student fnasir1@charlotte.edu"
}

ticker = "GOOGL"
cik = "0001652044"

url = f"https://data.sec.gov/submissions/CIK{cik}.json"
response = requests.get(url, headers=headers, timeout=30)

print("HTTP status:", response.status_code)

google_records = []

if response.status_code == 200:
    data = response.json()
    recent = data["filings"]["recent"]

    for form, date, accession, document in zip(
        recent["form"],
        recent["filingDate"],
        recent["accessionNumber"],
        recent["primaryDocument"]
    ):
        if form in ["10-K", "10-Q"] and date.startswith("2022"):
            google_records.append({
                "ticker": ticker,
                "company": data["name"],
                "form": form,
                "filing_date": date,
                "accession_number": accession,
                "primary_document": document
            })

google_df = pd.DataFrame(google_records)

print("Google filings found:", len(google_df))
display(google_df)


HTTP status: 200
Google filings found: 0


""


In [30]:

import requests
import pandas as pd

headers = {
    "User-Agent": "MarketStressResearch Student fnasir1@charlotte.edu"
}

cik = "0001652044"
url = f"https://data.sec.gov/submissions/CIK{cik}.json"

response = requests.get(url, headers=headers, timeout=30)
data = response.json()

filing_history_files = data.get("filings", {}).get("files", [])

print("Historical filing index files:", len(filing_history_files))

for item in filing_history_files:
    print(item["name"], item.get("filingFrom"), item.get("filingTo"))


Historical filing index files: 1
CIK0001652044-submissions-001.json 2015-10-02 2023-07-17


## 9. Conclusion and Next Steps

This notebook collected and processed SEC filings for five major technology companies during the 2022 market stress period.

### Key Findings

- Successfully retrieved 20 SEC filings from EDGAR.
- Included NVIDIA, Apple, Alphabet, Microsoft, and Amazon.
- Extracted 13,217 text segments from the filing documents.
- Confirmed that all five companies are represented in the processed dataset.
- Found no missing or empty text segments during the final completeness check.
- Saved the processed text to `sec_text_segments_2022.csv`.

### Limitations

The dataset covers only five technology companies and does not represent the entire U.S. equity market.

Some extracted text contains filing headers, checkbox symbols, and other formatting artifacts. Additional preprocessing may be required before applying sentiment analysis models.

SEC filings are published periodically and may not capture rapidly changing market sentiment.

### Next Steps

1. Investigate financial-news data sources and their access limitations.
2. Compare SEC filings alone with a combined SEC filings and financial-news approach.
3. Recommend a suitable text source for the Market Stress Early-Warning System.
4. Apply financial sentiment analysis methods in a later project stage.
